SETUP - Data preparation

In [1]:
from __future__ import annotations

from datetime import datetime
from collections.abc import Iterator
from dateutil.relativedelta import relativedelta
import joblib
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, RandomizedSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix
from scipy.stats import loguniform, randint
from sqlalchemy import create_engine, text
import os
import time

# ========== Runtime constants ==========
DB_CHUNK_SIZE   = 100_000  # rows per SQL chunk in BloodTests streaming download
DB_MAX_RETRIES  = 3        # max retry attempts on transient DB/network errors
SEARCH_N_ITER   = 15       # RandomizedSearchCV iterations (increase for longer tuning)

# ========== Connection ==========
# Create a SQLAlchemy engine to the MySQL database.
# pool_pre_ping=True avoids "stale" connections by pinging before checkout.

HOST = os.getenv("DB_HOST")
USER = os.getenv("DB_USER")
PWD  = os.getenv("DB_PASSWORD")
DB   = os.getenv("DB_SCHEMA")

if not all([HOST, USER, PWD, DB]):
    raise EnvironmentError(
        "Missing DB credentials. Copy .env.example to .env and fill in the values."
    )

engine = create_engine(
    f"mysql+pymysql://{USER}:{PWD}@{HOST}/{DB}",
    pool_pre_ping=True,
    connect_args={"connect_timeout": 15, "read_timeout": 900, "write_timeout": 900},
)

# ========== BloodTests Table "tests" to keep (pandas filter) ==========
# Only these lab tests (relevant for smoker-drinker assesment) will be kept and pivoted later (others are dropped).
TESTS_KEEP = {
  'blood glucose level', 'blood serum creatinine', 'cholesterol (total)',
  'diastolic blood pressure', 'gamma glutamyl transferase', 'hemoglobin',
  'high density lipoprotein cholesterol', 'low density lipoprotein cholesterol',
  'proteinuria', 'serum glutamic oxaloacetic transaminase (alt)',
  'serum glutamic oxaloacetic transaminase (ast)',
  'systolic blood pressure', 'triglycerides level'
}

# Yield consecutive monthly windows [start, next) in YYYY-MM-DD strings for later BloodTests queries.
# Example: ('2023-01-01', '2023-02-01'), then ('2023-02-01','2023-03-01'), ...
def month_windows(start: datetime, end: datetime) -> Iterator[tuple[str, str]]:
    cur = datetime(start.year, start.month, 1)
    while cur <= end:
        nxt = cur + relativedelta(months=1)
        yield cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d")
        cur = nxt

# ========== 1) BloodTests date range ==========
# Find overall min/max dates available in the BloodTests table to bound monthly downloads.
with engine.connect() as conn:
    dminmax = pd.read_sql_query(
        text("SELECT MIN(lab_test_date) AS dmin, MAX(lab_test_date) AS dmax FROM BloodTests"),
        conn
    ).iloc[0]
dmin = pd.to_datetime(dminmax["dmin"], errors="coerce")
dmax = pd.to_datetime(dminmax["dmax"], errors="coerce")
# Fallback wide range if dates are missing/null.
if pd.isna(dmin) or pd.isna(dmax):
    dmin = datetime(1970,1,1); dmax = datetime(2100,1,1)
print(f"[INFO] BloodTests date range: {dmin} → {dmax}")

START = dmin   # e.g., datetime(2023,1,1)
END   = dmax

# ========== 2) Download BloodTests by month ==========
# Stream monthly chunks to control memory usage and add basic cleaning.
SQL_BT_MONTH = text("""
SELECT
  session_id AS device_id,
  test,
  result,
  lab_test_date
FROM BloodTests
WHERE lab_test_date >= :d_from
  AND lab_test_date <  :d_to
""")

t0 = time.time()
bt_chunks = []
with engine.connect().execution_options(stream_results=True) as conn:
    for d_from, d_to in month_windows(START, END):
        tries = 0
        while True:
            try:
                tic = time.time()
                for chunk in pd.read_sql_query(SQL_BT_MONTH, conn,
                                               params={"d_from": d_from, "d_to": d_to},
                                               chunksize=DB_CHUNK_SIZE):
                    # Standardize and type-cast: trim/lower test name, numeric result, datetime test date.
                    chunk["test"] = chunk["test"].astype(str).str.strip().str.lower()
                    chunk["result"] = pd.to_numeric(chunk["result"], errors="coerce")
                    chunk["lab_test_date"] = pd.to_datetime(chunk["lab_test_date"], errors="coerce")
                    bt_chunks.append(chunk)
                break
            except Exception as e:
                # Simple retry with linear backoff (up to 3 attempts) to handle transient DB/network issues.
                tries += 1
                if tries >= DB_MAX_RETRIES:
                    raise
                wait = 3*tries
                print(f"[BT] retry {tries} ({e}) — wait {wait}s")
                time.sleep(wait)

# Concatenate all monthly chunks into a single DataFrame (or an empty schema if none fetched).
bt = (pd.concat(bt_chunks, ignore_index=True)
      if bt_chunks else pd.DataFrame(columns=["device_id","test","result","lab_test_date"]))
print(f"[INFO] BloodTests rows fetched: {len(bt)} | elapsed {time.time()-t0:.1f}s")

# ========== 3) Filter tests -> take latest per (device_id, test) -> pivot ==========
# Keep only selected tests, sort by date, take the last measurement per device & test.
bt = bt[bt["test"].isin(TESTS_KEEP)]
bt = bt.sort_values(["device_id","test","lab_test_date"])
latest = bt.groupby(["device_id","test"], as_index=False).last()

# Wide pivot: one row per device_id, columns are test names (prefixed with bt_ and sanitized).
bt_wide = latest.pivot(index="device_id", columns="test", values="result")
bt_wide.columns = [f"bt_{c.replace(' ', '_').replace('(', '').replace(')', '')}" for c in bt_wide.columns]
bt_wide = bt_wide.reset_index()
bt_wide["device_id"] = bt_wide["device_id"].astype(str)
print("[INFO] bt_wide:", bt_wide.shape)

# ========== 4) Main Step2 ==========
# Build a cleaned demographic/administrative table from Main:
# - Normalize birth_year (2-digit to 1900s/2000s, keep 4-digit) -> birth_year_fixed
# - Derive age as 2025 - birth_year_fixed
# - Normalize state into major ones {Ontario, Quebec, British Columbia, Alberta, Other}
SQL_MAIN_STEP2 = text("""
SELECT
  m.device_id,
  m.birth_year                                              AS birth_year_original,
  CASE
    WHEN by_num IS NULL                      THEN NULL
    WHEN by_num >= 1900                     THEN by_num
    WHEN by_num BETWEEN 0  AND 25           THEN 2000 + by_num
    WHEN by_num BETWEEN 26 AND 99           THEN 1900 + by_num
    ELSE NULL
  END AS birth_year_fixed,
  CASE
    WHEN (
      CASE
        WHEN by_num IS NULL                      THEN NULL
        WHEN by_num >= 1900                     THEN by_num
        WHEN by_num BETWEEN 0  AND 25           THEN 2000 + by_num
        WHEN by_num BETWEEN 26 AND 99           THEN 1900 + by_num
        ELSE NULL
      END
  ) IS NULL THEN NULL
    ELSE 2025 - (
      CASE
        WHEN by_num IS NULL                      THEN NULL
        WHEN by_num >= 1900                     THEN by_num
        WHEN by_num BETWEEN 0  AND 25           THEN 2000 + by_num
        WHEN by_num BETWEEN 26 AND 99           THEN 1900 + by_num
        ELSE NULL
      END
    )
  END AS age,
  m.gender,
  CASE
    WHEN m.state IS NULL OR TRIM(m.state) = ''             THEN 'Other'
    WHEN LOWER(TRIM(m.state)) = 'ontario'                  THEN 'Ontario'
    WHEN LOWER(TRIM(m.state)) = 'quebec'                   THEN 'Quebec'
    WHEN LOWER(TRIM(m.state)) = 'british columbia'         THEN 'British Columbia'
    WHEN LOWER(TRIM(m.state)) = 'alberta'                  THEN 'Alberta'
    ELSE 'Other'
  END AS state,
  m.smoker,
  m.drinker
FROM (
  SELECT
    device_id, birth_year, gender, state, smoker, drinker,
    CASE
      WHEN birth_year IS NULL OR TRIM(CAST(birth_year AS CHAR)) = '' THEN NULL
      WHEN TRIM(CAST(birth_year AS CHAR)) REGEXP '^[0-9]+(\\.[0-9]+)?$'
        THEN FLOOR(CAST(TRIM(CAST(birth_year AS CHAR)) AS DECIMAL(10,2)))
      ELSE NULL
    END AS by_num
  FROM Main
) AS m
""")
with engine.connect() as conn:
    main_step2 = pd.read_sql_query(SQL_MAIN_STEP2, conn)
main_step2["device_id"] = main_step2["device_id"].astype(str)
print("[INFO] main_step2:", main_step2.shape)

# ========== 4b) Physiotherapy: only device_id from Main; columns: waistline_in_cm, bmi ==========
# Simple SELECT; filtering and aggregation are done in pandas below.
SQL_PHYS = text("""
SELECT
  anonymous_mode_fingerprint AS device_id,
  waistline_in_cm,
  bmi
FROM Physiotherapy
""")
with engine.connect() as conn:
    phys = pd.read_sql_query(SQL_PHYS, conn)

# Keep only device_id that exist in main_step2 (inner domain alignment).
phys["device_id"] = phys["device_id"].astype(str)
phys = phys[phys["device_id"].isin(set(main_step2["device_id"]))]

# Cast to numeric and aggregate by device_id using the median if more then one value per device_id exists (robust to outliers).
for c in ["waistline_in_cm", "bmi"]:
    if c in phys.columns:
        phys[c] = pd.to_numeric(phys[c], errors="coerce")

phys_agg = phys.groupby("device_id", as_index=False).median(numeric_only=True)

# Basic plausibility cleaning on aggregated values.
if "bmi" in phys_agg.columns:
    phys_agg.loc[(phys_agg["bmi"] <= 0) | (phys_agg["bmi"] > 100), "bmi"] = pd.NA
if "waistline_in_cm" in phys_agg.columns:
    phys_agg.loc[(phys_agg["waistline_in_cm"] <= 30) | (phys_agg["waistline_in_cm"] > 250), "waistline_in_cm"] = pd.NA

print("[INFO] phys_agg:", phys_agg.shape)

# ========== 5) Final merge ==========
# Join demographics (Main Step2), latest lab tests (bt_wide), and physiotherapy medians on device_id.
df = (main_step2.merge(bt_wide, on="device_id", how="left")
                 .merge(phys_agg, on="device_id", how="left"))
print("[INFO] df:", df.shape)


[INFO] BloodTests date range: 2022-09-19 00:00:00 → 2025-09-20 00:00:00
[INFO] BloodTests rows fetched: 5019090 | elapsed 253.5s
[INFO] bt_wide: (415144, 14)
[INFO] main_step2: (432442, 8)
[INFO] phys_agg: (389197, 3)
[INFO] df: (432442, 23)


SETUP - Split dataset into Training and Classification + Create Skeleton

In [2]:
# =======================
# 0) Basic sanity checks
# =======================
assert "df" in globals(), "df not found: run the query/merge step first"
assert {"device_id","smoker","drinker"}.issubset(df.columns), "df must contain device_id, smoker, drinker"

# =======================
# 1) Normalization & mapping (dataset-specific)
# =======================
def _norm_str(s: pd.Series) -> pd.Series:
    # Normalize strings to lower-case and strip spaces to unify token matching
    return s.astype(str).str.strip().str.lower()

# Canonical tokens observed in the dataset + common aliases
SMOKE_YES = {
    "smoker", "smokes", "yes", "y", "true", "t", "1", "cigarettes are my only friend", "i can quit anytime i want!", "cigares do not kill people, people kill people"
}
SMOKE_NO = {
    "does not smoke", "non-smoker", "nonsmoker", "no", "n", "false", "f", "0"
}

DRINK_YES = {
    "drinks alcohol", "drinker", "yes", "y", "true", "t", "1", "yes but only johnny walker black label", "uuhh... i only drink on weekends"
}
DRINK_NO = {
    "does not drink", "non-drinker", "nondrinker", "no", "n", "false", "f", "0"
}

def is_known_smoker(series: pd.Series) -> pd.Series:
    # Known if the normalized token is in either YES or NO sets
    s = _norm_str(series)
    return s.isin(SMOKE_YES | SMOKE_NO)

def is_known_drinker(series: pd.Series) -> pd.Series:
    # Known if the normalized token is in either YES or NO sets
    d = _norm_str(series)
    return d.isin(DRINK_YES | DRINK_NO)

def to01_smoker(series: pd.Series) -> pd.Series:
    # Map tokens to {1,0}; if token is numeric-like, try converting it; return pandas Int64 (nullable)
    s = _norm_str(series)
    out = pd.Series(np.nan, index=series.index, dtype="float64")
    out[s.isin(SMOKE_YES)] = 1
    out[s.isin(SMOKE_NO)]  = 0
    rem = out.isna()
    if rem.any():
        out.loc[rem] = pd.to_numeric(s[rem], errors="coerce")
    return out.astype("Int64")

def to01_drinker(series: pd.Series) -> pd.Series:
    # Map tokens to {1,0}; if token is numeric-like, try converting it; return pandas Int64 (nullable)
    s = _norm_str(series)
    out = pd.Series(np.nan, index=series.index, dtype="float64")
    out[s.isin(DRINK_YES)] = 1
    out[s.isin(DRINK_NO)]  = 0
    rem = out.isna()
    if rem.any():
        out.loc[rem] = pd.to_numeric(s[rem], errors="coerce")
    return out.astype("Int64")

# Split diagnostics
print("[DBG] smoker tokens:", _norm_str(df["smoker"]).value_counts().head(10).to_dict())
print("[DBG] drinker tokens:", _norm_str(df["drinker"]).value_counts().head(10).to_dict())

# =======================
# 2) Split df (raw, post-merge)
# =======================
smoke_known = is_known_smoker(df["smoker"])
drink_known = is_known_drinker(df["drinker"])

# Training: both labels known; Classification: at least one label unknown
mask_training = smoke_known & drink_known
mask_classification = ~mask_training

df_training = df.loc[mask_training].copy()       
df_classification = df.loc[mask_classification].copy()

print("[SETUP] Total df:", len(df),
      "| Training:", len(df_training),
      "| Classification:", len(df_classification))

def _pct(x): return f"{100.0*float(x):.1f}%"
print("[SETUP] Label completeness on df:")
print("  smoker known:", _pct(smoke_known.mean()))
print("  drinker known:", _pct(drink_known.mean()))
print("  training share:", _pct(mask_training.mean()))
print("  classification share:", _pct(mask_classification.mean()))

# 0/1 encodings for training (diagnostics only; no imputations here)
df_training["smoker_01"]  = to01_smoker(df_training["smoker"])
df_training["drinker_01"] = to01_drinker(df_training["drinker"])

# =======================
# 3) Outputs for downstream steps
# =======================
out_dir = Path("./outputs_setup")
out_dir.mkdir(parents=True, exist_ok=True)

# 3a) CSV with device_id of the Classification Set (unique)
csv_path = out_dir / "classification_device_ids.csv"
df_classification[["device_id"]].drop_duplicates().to_csv(csv_path, index=False)
print(f"[SETUP] Saved:", csv_path)

# 3b) JSON skeleton for the Classification Set
#     These fields will be filled by later steps (predictions, advertiser decisions, persona)
json_skeleton_path = out_dir / "classification_skeleton.json"
skeleton = []
for dev in df_classification["device_id"].astype(str).drop_duplicates():
    skeleton.append({
        "device_id": dev,
        "user_drinks": None,     # (0/1)
        "user_smokes": None,     # (0/1)
        "johnny_talker": None,   # (0/1)
        "giggolo": None,         # (0/1)
        "vapeshape": None,       # (0/1)
        "persona": None          # "saint" | "fumer" | "tippler" | "sinner"
    })
with open(json_skeleton_path, "w", encoding="utf-8") as f:
    json.dump(skeleton, f, ensure_ascii=False, indent=2)
print(f"[SETUP] Saved:", json_skeleton_path)


[DBG] smoker tokens: {'does not smoke': 314540, 'smoker': 111923, 'none': 5976, 'cigarettes are my only friend': 1, 'i can quit anytime i want!': 1, 'cigares do not kill people, people kill people': 1}
[DBG] drinker tokens: {'does not drink': 230397, 'drinks alcohol': 196034, 'none': 6009, 'yes but only johnny walker black label': 1, 'uuhh... i only drink on weekends': 1}
[SETUP] Total df: 432442 | Training: 422442 | Classification: 10000
[SETUP] Label completeness on df:
  smoker known: 98.6%
  drinker known: 98.6%
  training share: 97.7%
  classification share: 2.3%
[SETUP] Saved: outputs_setup/classification_device_ids.csv
[SETUP] Saved: outputs_setup/classification_skeleton.json


SETUP - Cleaning & Imputation

In [3]:
# Numeric-like columns used in both datasets (will be coerced to numeric)
NUM_LIKE = [
    "birth_year_fixed", "age", "bmi", "waistline_in_cm",
    "bt_blood_glucose_level", "bt_blood_serum_creatinine",
    "bt_cholesterol_total", "bt_diastolic_blood_pressure",
    "bt_gamma_glutamyl_transferase", "bt_hemoglobin",
    "bt_high_density_lipoprotein_cholesterol",
    "bt_low_density_lipoprotein_cholesterol",
    "bt_proteinuria",
    "bt_sgot_alt", "bt_sgot_ast",
    "bt_systolic_blood_pressure", "bt_triglycerides_level"
]

# Biomarker candidates
BIOMARKER_CANDIDATES = [
    "bt_systolic_blood_pressure",
    "bt_diastolic_blood_pressure",
    "bt_blood_serum_creatinine",
    "bt_blood_glucose_level",
    "bt_cholesterol_total",
    "bt_high_density_lipoprotein_cholesterol",
    "bt_low_density_lipoprotein_cholesterol",
    "bt_triglycerides_level",
    "bt_gamma_glutamyl_transferase",
    "bt_sgot_alt",
    "bt_sgot_ast",
    "bt_hemoglobin",
    "bt_proteinuria",
]

def _force_numeric(df, cols):
    # Coerce listed columns to numeric (invalid values become NaN)
    for c in [c for c in cols if c in df.columns]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def _plausibility_checks(df):
    # Basic plausibility guards on anthropometrics and vitals
    if "age" in df.columns:
        df.loc[(df["age"] < 10) | (df["age"] > 100), "age"] = np.nan
    if "bmi" in df.columns:
        df.loc[(df["bmi"] <= 0) | (df["bmi"] > 100), "bmi"] = np.nan
    if "waistline_in_cm" in df.columns:
        df.loc[(df["waistline_in_cm"] <= 30) | (df["waistline_in_cm"] > 250), "waistline_in_cm"] = np.nan

    # Blood pressure: range checks + swap if diastolic > systolic (likely inverted)
    if {"bt_systolic_blood_pressure", "bt_diastolic_blood_pressure"}.issubset(df.columns):
        df.loc[(df["bt_systolic_blood_pressure"] < 60) | 
               (df["bt_systolic_blood_pressure"] > 300), "bt_systolic_blood_pressure"] = np.nan
        df.loc[(df["bt_diastolic_blood_pressure"] < 30) | 
               (df["bt_diastolic_blood_pressure"] > 200), "bt_diastolic_blood_pressure"] = np.nan
        mask_swap = (df["bt_systolic_blood_pressure"].notna() &
                     df["bt_diastolic_blood_pressure"].notna() &
                     (df["bt_diastolic_blood_pressure"] > df["bt_systolic_blood_pressure"]))
        if mask_swap.any():
            df.loc[mask_swap, ["bt_systolic_blood_pressure","bt_diastolic_blood_pressure"]] = \
                df.loc[mask_swap, ["bt_diastolic_blood_pressure","bt_systolic_blood_pressure"]].to_numpy()

    # Clinical values should not be negative
    for c in ["bt_cholesterol_total",
              "bt_high_density_lipoprotein_cholesterol",
              "bt_low_density_lipoprotein_cholesterol",
              "bt_triglycerides_level",
              "bt_blood_serum_creatinine",
              "bt_blood_glucose_level",
              "bt_gamma_glutamyl_transferase",
              "bt_hemoglobin"]:
        if c in df.columns:
            df.loc[df[c] < 0, c] = np.nan
    return df

def _ensure_device_id_unique(df, who="set"):
    # Ensure device_id is string and unique; keep the first occurrence
    df["device_id"] = df["device_id"].astype(str)
    dups = df["device_id"][df["device_id"].duplicated()].unique()
    if len(dups):
        print(f"[WARN] duplicate device_id in {who}: {len(dups)} — keeping first occurrence")
        df = df.drop_duplicates(subset=["device_id"], keep="first")
    else:
        print(f"[OK] unique device_id in {who}.")
    return df

def _add_age_if_missing(df):
    # If 'age' is missing but birth_year_fixed is available, derive age relative to 2025
    if "age" not in df.columns and "birth_year_fixed" in df.columns:
        df["age"] = 2025 - pd.to_numeric(df["birth_year_fixed"], errors="coerce")
    return df

def _make_age_bin(df):
    # Create decade bins for age: 10–19, 20–29, ..., 90–99
    bins = list(range(10, 101, 10))           # 10-19, 20-29, ...
    labels = [f"{i}-{i+9}" for i in bins[:-1]]
    df["age_bin"] = pd.cut(df["age"], bins=bins, labels=labels, right=False)
    return df

def _make_waist_bin_training(df):
    # Build waistline bins on TRAINING:
    # try quantile-based bins (qcut); if not feasible, fall back to fixed cut points
    if "waistline_in_cm" in df.columns:
        try:
            df["waist_bin"] = pd.qcut(df["waistline_in_cm"], q=5, duplicates="drop")
        except ValueError:
            bins_w = [40, 60, 80, 100, 120, 140, 200]
            df["waist_bin"] = pd.cut(df["waistline_in_cm"], bins=bins_w, right=False)
    else:
        df["waist_bin"] = pd.NA
    return df

def _apply_training_waist_bins(df_cls, df_trn):
    """Apply in CLASSIFICATION the same waist cut points learned on TRAINING."""
    if "waist_bin" in df_trn.columns and hasattr(df_trn["waist_bin"], "cat"):
        try:
            wbins = df_trn["waist_bin"].cat.categories
            edges = sorted({wb.left for wb in wbins}.union({wbins[-1].right}))
            df_cls["waist_bin"] = pd.cut(df_cls["waistline_in_cm"], bins=edges, right=False)
            return df_cls
        except Exception:
            pass
    # Fixed fallback if training bins are unavailable
    bins_w = [40, 60, 80, 100, 120, 140, 200]
    df_cls["waist_bin"] = pd.cut(df_cls["waistline_in_cm"], bins=bins_w, right=False)
    return df_cls

def _impute_contextual_training(df_in, col, keys_lvl1, keys_lvl2):
    """Impute a numeric column in TRAINING using group medians (lvl1 then lvl2) with global median fallback."""
    df_out = df_in
    med1 = (df_out.groupby(keys_lvl1)[col]
            .median().reset_index().rename(columns={col: f"{col}_med1"}))
    df_out = df_out.merge(med1, on=keys_lvl1, how="left")
    df_out[col] = df_out[col].fillna(df_out[f"{col}_med1"])
    df_out.drop(columns=[f"{col}_med1"], inplace=True)

    rem = df_out[col].isna()
    if rem.any():
        med2 = (df_out.groupby(keys_lvl2)[col]
                .median().reset_index().rename(columns={col: f"{col}_med2"}))
        df_out = df_out.merge(med2, on=keys_lvl2, how="left")
        df_out.loc[rem, col] = df_out.loc[rem, f"{col}_med2"]
        df_out.drop(columns=[f"{col}_med2"], inplace=True)

    df_out[col] = df_out[col].fillna(df_out[col].median(skipna=True))
    return df_out

def _impute_contextual_from_training(df_cls, df_trn, col, keys_lvl1, keys_lvl2):
    """Impute a numeric column in CLASSIFICATION using ONLY medians computed on TRAINING (distribution lock)."""
    # Precompute medians from TRAINING
    trn_num = pd.to_numeric(df_trn[col], errors="coerce")
    med1 = (pd.concat([df_trn[keys_lvl1], trn_num], axis=1)
            .groupby(keys_lvl1)[col].median().reset_index().rename(columns={col: f"{col}_med1"}))
    med2 = (pd.concat([df_trn[keys_lvl2], trn_num], axis=1)
            .groupby(keys_lvl2)[col].median().reset_index().rename(columns={col: f"{col}_med2"}))

    df_out = df_cls.merge(med1, on=keys_lvl1, how="left")
    df_out[col] = pd.to_numeric(df_out[col], errors="coerce")
    df_out[col] = df_out[col].fillna(df_out[f"{col}_med1"])
    df_out.drop(columns=[f"{col}_med1"], inplace=True)

    rem = df_out[col].isna()
    if rem.any():
        df_out = df_out.merge(med2, on=keys_lvl2, how="left")
        df_out.loc[rem, col] = df_out.loc[rem, f"{col}_med2"]
        df_out.drop(columns=[f"{col}_med2"], inplace=True)

    # Final fallback: global median computed on TRAINING
    df_out[col] = df_out[col].fillna(trn_num.median(skipna=True))
    return df_out

# ---------- 1) Common cleaning (training/classification) ----------
print(f"[CLEAN] Cleaning df_training: {len(df_training)} rows")
df_training = df_training.copy()
df_training = _force_numeric(df_training, NUM_LIKE)
df_training = _plausibility_checks(df_training)
df_training = _ensure_device_id_unique(df_training, "training")

print(f"[CLEAN] Cleaning df_classification: {len(df_classification)} rows")
df_classification = df_classification.copy()
df_classification = _force_numeric(df_classification, NUM_LIKE)
df_classification = _plausibility_checks(df_classification)
df_classification = _ensure_device_id_unique(df_classification, "classification")

# ---------- 2) Diagnostics (optional) ----------
# null_report_tr = (df_training.isna().mean().sort_values(ascending=False).rename("null_ratio").to_frame())
# print("\n[NULL REPORT - TRAINING] top 20:"); print(null_report_tr.head(20))
# num_present_tr = [c for c in NUM_LIKE if c in df_training.columns]
# if num_present_tr:
#    print("\n[STATS TRAINING] numeric (post-clean):")
#    print(df_training[num_present_tr].describe(percentiles=[0.01, 0.5, 0.99]).T)

# ---------- 3) Contextual imputation — TRAINING ----------
df_training_ml = df_training.copy()
df_training_ml = _add_age_if_missing(df_training_ml)
df_training_ml = _make_age_bin(df_training_ml)
df_training_ml = _make_waist_bin_training(df_training_ml)

# Contextual imputation for BMI and waistline on TRAINING (grouped medians by gender x age_bin)
for col in ["bmi", "waistline_in_cm"]:
    if col in df_training_ml.columns:
        grp = (df_training_ml.groupby(["gender", "age_bin"])[col]
               .median().reset_index().rename(columns={col: f"{col}_median"}))
        df_training_ml = df_training_ml.merge(grp, on=["gender", "age_bin"], how="left")
        df_training_ml[col] = df_training_ml[col].fillna(df_training_ml[f"{col}_median"])
        df_training_ml[col] = df_training_ml[col].fillna(df_training_ml[col].median(skipna=True))
        df_training_ml.drop(columns=[f"{col}_median"], inplace=True)

# Contextual imputation for biomarkers on TRAINING (lvl1: gender,age_bin,waist_bin -> lvl2: gender,age_bin)
lvl1 = ["gender", "age_bin", "waist_bin"]
lvl2 = ["gender", "age_bin"]
biomarkers_tr = [c for c in BIOMARKER_CANDIDATES if c in df_training_ml.columns]
for col in biomarkers_tr:
    df_training_ml[col] = pd.to_numeric(df_training_ml[col], errors="coerce")
    df_training_ml = _impute_contextual_training(df_training_ml, col, lvl1, lvl2)

# Categorical basics (TRAINING): impute common categories; targets can be imputed here if desired
for c in ["gender", "state", "smoker", "drinker"]:
    if c in df_training_ml.columns:
        mode = df_training_ml[c].mode(dropna=True)
        df_training_ml[c] = df_training_ml[c].fillna(mode.iloc[0] if len(mode) else "Unknown")

# ---------- 4) Contextual imputation — CLASSIFICATION ----------
df_classification_ml = df_classification.copy()
df_classification_ml = _add_age_if_missing(df_classification_ml)
df_classification_ml = _make_age_bin(df_classification_ml)
df_classification_ml = _apply_training_waist_bins(df_classification_ml, df_training_ml)

# Impute BMI/waistline in CLASSIFICATION using ONLY TRAINING medians (gender x age_bin)
for col in ["bmi", "waistline_in_cm"]:
    if col in df_classification_ml.columns:
        df_classification_ml = _impute_contextual_from_training(
            df_classification_ml, df_training_ml, col, keys_lvl1=["gender","age_bin"], keys_lvl2=["gender","age_bin"]
        )

# Impute biomarkers in CLASSIFICATION using ONLY TRAINING medians (distribution preserved)
biomarkers_cls = [c for c in BIOMARKER_CANDIDATES if c in df_classification_ml.columns and c in df_training_ml.columns]
for col in biomarkers_cls:
    df_classification_ml = _impute_contextual_from_training(df_classification_ml, df_training_ml, col, lvl1, lvl2)

# Categorical basics in CLASSIFICATION
for c in ["gender", "state"]:
    if c in df_classification_ml.columns:
        mode_tr = df_training_ml[c].mode(dropna=True)
        df_classification_ml[c] = df_classification_ml[c].fillna(mode_tr.iloc[0] if len(mode_tr) else "Unknown")

# ---------- 5) Diagnostics (optional) ----------
# null_report_cls = (df_classification_ml.isna().mean().sort_values(ascending=False).rename("null_ratio").to_frame())
# print("\n[NULL REPORT - CLASSIFICATION] top 20:"); print(null_report_cls.head(20))
# num_present_cls = [c for c in NUM_LIKE if c in df_classification_ml.columns]
# if num_present_cls:
#    print("\n[STATS CLASSIFICATION] numeric (post-clean):")
#    print(df_classification_ml[num_present_cls].describe(percentiles=[0.01, 0.5, 0.99]).T)

# ---------- 6) Summary ----------
print(f"\n[SHARE] Training: {len(df_training)} ({len(df_training)/len(df)*100:.1f}%) "
      f"| Classification: {len(df_classification)} ({len(df_classification)/len(df)*100:.1f}%)")


[CLEAN] Cleaning df_training: 422442 rows
[OK] unique device_id in training.
[CLEAN] Cleaning df_classification: 10000 rows
[OK] unique device_id in classification.


/var/folders/_n/h73_n7md2fb62fnnsr1x3zb00000gn/T/ipykernel_54510/2623175774.py:198: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grp = (df_training_ml.groupby(["gender", "age_bin"])[col]
/var/folders/_n/h73_n7md2fb62fnnsr1x3zb00000gn/T/ipykernel_54510/2623175774.py:198: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grp = (df_training_ml.groupby(["gender", "age_bin"])[col]
/var/folders/_n/h73_n7md2fb62fnnsr1x3zb00000gn/T/ipykernel_54510/2623175774.py:127: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current 


[SHARE] Training: 422442 (97.7%) | Classification: 10000 (2.3%)


/var/folders/_n/h73_n7md2fb62fnnsr1x3zb00000gn/T/ipykernel_54510/2623175774.py:149: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(keys_lvl1)[col].median().reset_index().rename(columns={col: f"{col}_med1"}))
/var/folders/_n/h73_n7md2fb62fnnsr1x3zb00000gn/T/ipykernel_54510/2623175774.py:151: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(keys_lvl2)[col].median().reset_index().rename(columns={col: f"{col}_med2"}))


Q1 - Train HGB models + tune + choose accuracy-optimal thresholds + predict LT, C...

In [4]:
# =======================
# 0) Preconditions & dataframe selection
# =======================
# Ensure training exists; ensure at least one classification dataframe exists.
assert "df_training_ml" in globals(), "df_training_ml not found: please perform the cleaning step on the training first"
assert ("df_classification_ml" in globals()) or ("df_classification" in globals()), \
       "Neither df_classification_ml nor df_classification found: run split/clean first"

X_train = df_training_ml.copy()

# Prefer the CLEANED classification if present; otherwise use the raw one.
# (Pipelines below can minimally handle categorical imputations.)
if "df_classification_ml" in globals():
    X_cls = df_classification_ml.copy()
    print("[Q1.1] I use df_classification_ml (clean) for predictions")
else:
    X_cls = df_classification.copy()
    print("[Q1.1] WARNING: I'm using df_classification (raw). The pipeline will handle minimal imputations on the categories.")

print("[Q1.1] Training:", len(X_train), "| Classification:", len(X_cls))

# Targets come from training (known there).
y_smokes = to01_smoker(X_train["smoker"]).astype(int)
y_drinks = to01_drinker(X_train["drinker"]).astype(int)

# =======================
# 1) Feature set with ALT/AST alias handling
# =======================
def first_present(cols, pool):
    # Return the first column name from `cols` that exists in `pool`; else None.
    for c in cols:
        if c in pool:
            return c
    return None

# ALT/AST can have alternative names; pick whichever exists in the training columns.
ALT = first_present(
    ["bt_serum_glutamic_oxaloacetic_transaminase_alt", "bt_sgot_alt"],
    X_train.columns
)
AST = first_present(
    ["bt_serum_glutamic_oxaloacetic_transaminase_ast", "bt_sgot_ast"],
    X_train.columns
)

# Core features shared; plus task-specific extras
common_feats = [
    "age", "bmi", "waistline_in_cm",
    "gender", "state",
    "bt_high_density_lipoprotein_cholesterol",
    "bt_triglycerides_level",
    "bt_systolic_blood_pressure",
    "bt_diastolic_blood_pressure",
    "bt_blood_glucose_level",
    "bt_blood_serum_creatinine",
    "bt_cholesterol_total",
    "bt_proteinuria",
]
smoker_extra  = ["bt_low_density_lipoprotein_cholesterol", "bt_hemoglobin"]
drinker_extra = ["bt_gamma_glutamyl_transferase", AST, ALT]

# Remove any None left by missing aliases to avoid pipeline errors.
drinker_extra = [c for c in drinker_extra if c is not None]

# Keep only features actually present in TRAINING
features_smoker  = [c for c in (common_feats + smoker_extra)  if c in X_train.columns]
features_drinker = [c for c in (common_feats + drinker_extra) if c in X_train.columns]

# Separate numerical vs categorical (main categoricals are gender/state)
num_cols_smoker  = [c for c in features_smoker  if c not in ["gender","state"]]
cat_cols_smoker  = [c for c in ["gender","state"] if c in features_smoker]
num_cols_drinker = [c for c in features_drinker if c not in ["gender","state"]]
cat_cols_drinker = [c for c in ["gender","state"] if c in features_drinker]

print("[Q1.1] Feature smoker:", features_smoker)
print("[Q1.1] Feature drinker:", features_drinker)

# =======================
# 2) Preprocess: HGB tolerates numeric NaNs; for categoricals we impute mode + OHE
# =======================
def make_ohe() -> OneHotEncoder:
    """Return a dense OneHotEncoder compatible with both sklearn <1.2 (sparse=)
    and sklearn >=1.2 (sparse_output=)."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_pipeline(num_cols, cat_cols, random_state=0):
    # Column-wise preprocessing:
    # - numeric: passthrough (HGB can handle NaNs)
    # - categorical: impute most frequent, then one-hot encode (ignore unseen categories)
    pre = ColumnTransformer(
        transformers=[
            ("num", "passthrough", num_cols),
            ("cat", Pipeline([
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("ohe", make_ohe())
            ]), cat_cols),
        ],
        remainder="drop",
        n_jobs=None,
    )
    # HistGradientBoostingClassifier
    clf = HistGradientBoostingClassifier(
        learning_rate=0.1,
        max_depth=None,
        max_iter=300,
        min_samples_leaf=20,
        l2_regularization=0.0,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=random_state
    )
    # Unified pipeline: fit() does pre.fit_transform + clf.fit;
    # predict/predict_proba do pre.transform + clf.predict/ predict_proba.
    return Pipeline([("pre", pre), ("clf", clf)])

# =======================
# 3) Compact param search space (fast) + CV setup
# =======================
# Randomized search over key HGB hyperparameters (reasonable ranges)
param_dist = {
    "clf__learning_rate": loguniform(0.03, 0.3),
    "clf__max_iter": randint(200, 500),
    "clf__max_depth": [None, 6, 10],
    "clf__min_samples_leaf": randint(10, 60),
    "clf__l2_regularization": loguniform(1e-4, 10.0),
    "clf__max_bins": randint(128, 255),
}

# Stratified folds for balanced class proportions in each fold
cv5_smoke = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
cv5_drink = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

def fit_and_eval(
    X: pd.DataFrame,
    y: pd.Series,
    num_cols: list[str],
    cat_cols: list[str],
    cv5: StratifiedKFold,
    seed: int,
    label: str,
) -> tuple[Pipeline, dict]:
    # Build pipeline, then randomized search (3-fold) on top; refit on best params.
    base = make_pipeline(num_cols, cat_cols, random_state=seed)
    search = RandomizedSearchCV(
        estimator=base,
        param_distributions=param_dist,
        n_iter=SEARCH_N_ITER,      # small budget for speed
        scoring="accuracy",
        cv=3,
        verbose=1,
        n_jobs=-1,
        random_state=seed,
        refit=True
    )
    search.fit(X, y)
    print(f"[{label}] best params:", search.best_params_)
    best = search.best_estimator_

    # Robust evaluation with 5-fold:
    # - mean/stdev of CV accuracy
    # - OOF predictions for confusion matrix and global accuracy
    cv_scores = cross_val_score(best, X, y, cv=cv5, scoring="accuracy")
    oof_pred = cross_val_predict(best, X, y, cv=cv5, method="predict")
    acc = accuracy_score(y, oof_pred)
    cm  = confusion_matrix(y, oof_pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    print(f"[{label}] CV accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"[{label}] OOF acc=(TP+TN)/Total = {(tp+tn)/(tp+tn+fp+fn):.4f}")
    print(f"[{label}] Confusion matrix [[TN,FP],[FN,TP]]:\n{cm}")

    # Final fit on the entire training set (to deploy the best model)
    best.fit(X, y)
    return best, dict(
        cv_accuracy_mean=float(cv_scores.mean()),
        cv_accuracy_std=float(cv_scores.std()),
        oof_accuracy=float(acc),
        confusion_matrix=cm.tolist(),
        tp=int(tp), tn=int(tn), fp=int(fp), fn=int(fn),
    )

# =======================
# 4) Train & eval
# =======================
print("\n[Q1.1] ====== SMOKER ======")
best_smoke, m_smoke = fit_and_eval(X_train, y_smokes, num_cols_smoker, cat_cols_smoker, cv5_smoke, 11, "SMOKER")

print("\n[Q1.1] ====== DRINKER ======")
best_drink, m_drink = fit_and_eval(X_train, y_drinks, num_cols_drinker, cat_cols_drinker, cv5_drink, 22, "DRINKER")

# =======================
# 4b) Choose threshold that maximizes OOF accuracy for SMOKER/DRINKER
# =======================
def find_best_threshold(
    model: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    cv: StratifiedKFold,
    label: str = "",
) -> tuple[float, float, np.ndarray, float]:
    """
    Compute OOF probabilities with cross_val_predict and choose the threshold
    that maximizes accuracy. Returns (best_threshold, best_accuracy, proba_oof, acc_at_0p5).
    """
    proba_oof = cross_val_predict(model, X, y, cv=cv, method="predict_proba")[:, 1]
    # Candidate thresholds: all unique predicted probabilities (rounded) plus 0.5 for reference.
    thr_cand = np.unique(np.round(proba_oof, 4))
    thr_cand = np.unique(np.concatenate([thr_cand, [0.5]]))
    best_t, best_acc = 0.5, -1.0
    for t in thr_cand:
        pred = (proba_oof >= t).astype(int)
        acc = accuracy_score(y, pred)
        if acc > best_acc:
            best_t, best_acc = t, acc
    acc_at_05 = accuracy_score(y, (proba_oof >= 0.5).astype(int))
    print(f"[THRESH] {label}: best_t={best_t:.4f} | OOF acc*={best_acc:.4f} | acc@0.5={acc_at_05:.4f}")
    return best_t, best_acc, proba_oof, acc_at_05

# Find per-task optimal thresholds using the features used by each model
t_smoke, acc_smoke_best, proba_oof_smoke, acc_smoke_05 = find_best_threshold(
    best_smoke, X_train[features_smoker], y_smokes, cv5_smoke, label="SMOKER"
)
t_drink, acc_drink_best, proba_oof_drink, acc_drink_05 = find_best_threshold(
    best_drink, X_train[features_drinker], y_drinks, cv5_drink, label="DRINKER"
)

# =======================
# 5) Predictions on the Classification Set (using accuracy-optimal thresholds)
# =======================
# Ensure X_cls has all required features; create any missing ones as NaN (compatible with pipeline)
missing_smoker  = [c for c in features_smoker  if c not in X_cls.columns]
missing_drinker = [c for c in features_drinker if c not in X_cls.columns]
for c in set(missing_smoker + missing_drinker):
    X_cls[c] = np.nan

# Predict probabilities then apply task-specific thresholds
proba_smokes = best_smoke.predict_proba(X_cls[features_smoker])[:, 1]
pred_smokes  = (proba_smokes >= t_smoke).astype(int)

proba_drinks = best_drink.predict_proba(X_cls[features_drinker])[:, 1]
pred_drinks  = (proba_drinks >= t_drink).astype(int)

# Build output dataframe with IDs, hard predictions, and probabilities
df_preds = X_cls[["device_id"]].copy()
df_preds["pred_smokes"]  = pred_smokes
df_preds["proba_smokes"] = proba_smokes
df_preds["pred_drinks"]  = pred_drinks
df_preds["proba_drinks"] = proba_drinks

# =======================
# OVERRIDE with known values from Classification Set + update skeleton
# =======================

# 1) Build map of KNOWN values (0/1) in the classification set
#    - use existing helper functions: is_known_smoker/is_known_drinker, to01_smoker/to01_drinker
known_smoke_mask = is_known_smoker(df_classification["smoker"])
known_drink_mask = is_known_drinker(df_classification["drinker"])

df_known = df_classification[["device_id"]].copy()
df_known["device_id"] = df_known["device_id"].astype(str)

# SAFE build of known labels: never cast to int while NA is present
smoke01 = to01_smoker(df_classification["smoker"])   # expects 0/1 or NA
drink01 = to01_drinker(df_classification["drinker"]) # expects 0/1 or NA

# Use .where so unknowns stay NA; keep dtype float to allow NA
df_known["known_smokes"] = smoke01.where(known_smoke_mask).astype("float")
df_known["known_drinks"] = drink01.where(known_drink_mask).astype("float")


# Keep only rows where at least one target is known
df_known = df_known.dropna(how="all", subset=["known_smokes", "known_drinks"])

# 2) Align and OVERRIDE in df_preds (predictions and probabilities)
df_preds["device_id"] = df_preds["device_id"].astype(str)
df_merged = df_preds.merge(df_known, on="device_id", how="left")

# Override SMOKE where known
mask_sm = df_merged["known_smokes"].notna()
if mask_sm.any():
    # Prediction must equal the known value
    df_merged.loc[mask_sm, "pred_smokes"] = df_merged.loc[mask_sm, "known_smokes"].astype(int).values
    # Probability becomes 1.0 if known=1, 0.0 if known=0
    df_merged.loc[mask_sm, "proba_smokes"] = df_merged.loc[mask_sm, "known_smokes"].astype(float).values

# Override DRINK where known
mask_dr = df_merged["known_drinks"].notna()
if mask_dr.any():
    df_merged.loc[mask_dr, "pred_drinks"] = df_merged.loc[mask_dr, "known_drinks"].astype(int).values
    df_merged.loc[mask_dr, "proba_drinks"] = df_merged.loc[mask_dr, "known_drinks"].astype(float).values

# Replace df_preds with the corrected version (same columns as before)
df_preds = df_merged[["device_id", "pred_smokes", "proba_smokes", "pred_drinks", "proba_drinks"]].copy()

# 3) Update SETUP skeleton JSON:
#    - if there is a known 0/1 value -> keep it
#    - else, use the model prediction we just updated
sk_path = Path("./outputs_setup/classification_skeleton.json")
if sk_path.exists():
    with open(sk_path, "r", encoding="utf-8") as f:
        sk_list = json.load(f)
    df_sk = pd.DataFrame(sk_list)
    if "device_id" not in df_sk.columns:
        # Fallback (should not happen if SETUP ran correctly)
        df_sk["device_id"] = df_classification["device_id"].astype(str).drop_duplicates().values
    df_sk["device_id"] = df_sk["device_id"].astype(str)

    # Merge skeleton with known values and updated predictions
    sk = (
        df_sk
        .merge(df_known[["device_id","known_smokes","known_drinks"]], on="device_id", how="left")
        .merge(df_preds[["device_id","pred_smokes","pred_drinks"]], on="device_id", how="left")
    )

    def _choose(known_val, existing_val, pred_val):
        # Priority order:
        # 1) known value (0/1)
        # 2) existing value in skeleton (if already 0/1)
        # 3) model prediction
        # 4) None
        if pd.notna(known_val):
            return int(known_val)
        if existing_val in (0, 1):
            return int(existing_val)
        return int(pred_val) if pd.notna(pred_val) else None

    sk["user_smokes"] = [
        _choose(k, e, p) for k, e, p in zip(sk["known_smokes"], sk["user_smokes"], sk["pred_smokes"])
    ]
    sk["user_drinks"] = [
        _choose(k, e, p) for k, e, p in zip(sk["known_drinks"], sk["user_drinks"], sk["pred_drinks"])
    ]

    # Drop helper columns and write back to disk
    cols_drop = [c for c in ["known_smokes","known_drinks","pred_smokes","pred_drinks"] if c in sk.columns]
    sk.drop(columns=cols_drop, inplace=True)
    updated = sk.to_dict(orient="records")
    with open(sk_path, "w", encoding="utf-8") as f:
        json.dump(updated, f, ensure_ascii=False, indent=2)

    print(f"[Q1.2] Skeleton updated (known values override predictions): {sk_path}")
else:
    print("[Q1.2] Skeleton not found (SETUP not run yet).")

# =======================
# 6) Persistence (models, predictions, metrics)
# =======================
out = Path("./outputs_Q1")
out.mkdir(parents=True, exist_ok=True)

# Predictions for Q2
df_preds.to_csv(out / "classification_predictions.csv", index=False)

# Save trained models (can be reloaded for inference)
joblib.dump(best_smoke, out / "model_smoker_hgb.pkl")
joblib.dump(best_drink, out / "model_drinker_hgb.pkl")

# Save metrics and chosen thresholds for reproducibility/audit
metrics = {
    "smoker":  {
        "features_used": features_smoker,
        "best_threshold": float(t_smoke),
        "oof_best_accuracy": float(acc_smoke_best),
        "oof_accuracy_at_0.5": float(acc_smoke_05),
        **m_smoke
    },
    "drinker": {
        "features_used": features_drinker,
        "best_threshold": float(t_drink),
        "oof_best_accuracy": float(acc_drink_best),
        "oof_accuracy_at_0.5": float(acc_drink_05),
        **m_drink
    },
}
with open(out / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print(f"[Q1.1] Models, predictions and metrics saved in: {out}")



[Q1.1] I use df_classification_ml (clean) for predictions
[Q1.1] Training: 422442 | Classification: 10000
[Q1.1] Feature smoker: ['age', 'bmi', 'waistline_in_cm', 'gender', 'state', 'bt_high_density_lipoprotein_cholesterol', 'bt_triglycerides_level', 'bt_systolic_blood_pressure', 'bt_diastolic_blood_pressure', 'bt_blood_glucose_level', 'bt_blood_serum_creatinine', 'bt_cholesterol_total', 'bt_proteinuria', 'bt_low_density_lipoprotein_cholesterol', 'bt_hemoglobin']
[Q1.1] Feature drinker: ['age', 'bmi', 'waistline_in_cm', 'gender', 'state', 'bt_high_density_lipoprotein_cholesterol', 'bt_triglycerides_level', 'bt_systolic_blood_pressure', 'bt_diastolic_blood_pressure', 'bt_blood_glucose_level', 'bt_blood_serum_creatinine', 'bt_cholesterol_total', 'bt_proteinuria', 'bt_gamma_glutamyl_transferase', 'bt_serum_glutamic_oxaloacetic_transaminase_ast', 'bt_serum_glutamic_oxaloacetic_transaminase_alt']

[Q1.1] ====== SMOKER ======
Fitting 3 folds for each of 15 candidates, totalling 45 fits
[SMOK

Q2 - Personas, Advertiser Decisions & Expected Profits

In [5]:
# Paths:
# P1: holds classification_skeleton.json produced in SETUP
# P2: holds classification_predictions.csv (hard preds + probabilities) from Q1
# OUT: optional exports for final artifacts
P1 = Path("./outputs_setup")
P2 = Path("./outputs_Q1")
OUT = Path("./outputs_Q2") 
OUT.mkdir(parents=True, exist_ok=True)

# 1) Load SETUP skeleton (must already have user_smokes/user_drinks filled after Q1)
#    We validate presence and non-null targets before proceeding.
sk_path = P1 / "classification_skeleton.json"
assert sk_path.exists(), f"Skeleton not found: {sk_path}. Execute SETUP."
with open(sk_path, "r", encoding="utf-8") as f:
    sk_list = json.load(f)
df = pd.DataFrame(sk_list)
df["device_id"] = df["device_id"].astype(str)

# Sanity check: binary targets must be present and non-null; otherwise later persona logic cannot run.
for col in ["user_smokes", "user_drinks"]:
    assert col in df.columns, f"Missing column in skeleton: {col}"
    if df[col].isna().any():
        raise ValueError(
            f"Skeleton contains {col}=None. Make sure that after Q1 the skeleton has been updated with predictions."
        )

# 2) Load probabilities from Q1 CSV and merge into skeleton
#    We need proba_smokes/proba_drinks for expected value calculations and advertiser decisions.
preds_csv = P2 / "classification_predictions.csv"
assert preds_csv.exists(), f"Predictions CSV not found: {preds_csv}. Execute Q1."
df_preds = pd.read_csv(preds_csv)
df_preds["device_id"] = df_preds["device_id"].astype(str)

need_cols = {"device_id", "proba_smokes", "proba_drinks"}
missing = need_cols - set(df_preds.columns)
assert not missing, f"Missing coloumn in CSV: {missing}"

# Robustness: ensure unique device_id in predictions; if duplicates exist, keep first occurrence.
if df_preds["device_id"].duplicated().any():
    dup_n = df_preds["device_id"].duplicated().sum()
    print(f"[Q2][WARN] device_id duplicates in CSV predictions: {dup_n} — keeping first occurrence")
    df_preds = df_preds.drop_duplicates(subset=["device_id"], keep="first")

# Merge probabilities onto the working dataframe
df = df.merge(df_preds[["device_id", "proba_smokes", "proba_drinks"]], on="device_id", how="left")

# 3) Persona assignment (derived from binary targets)
#    Deterministic persona mapping based on (user_smokes, user_drinks).
def persona_from_targets(us: int, ud: int) -> str:
    if us == 0 and ud == 0: return "saint"
    if us == 1 and ud == 0: return "fumer"
    if us == 0 and ud == 1: return "tippler"
    return "sinner"  # us==1 and ud==1

df["persona"] = [persona_from_targets(int(us), int(ud)) for us, ud in zip(df["user_smokes"], df["user_drinks"])]

# 4) Price table per persona (from assignment specification)
#    Positive values = expected gain if we sell that user to the advertiser; negative = penalty/cost.
PRICES = {
    "johnny_talker": {"saint": -14, "sinner": 22, "fumer": -13, "tippler": 10},
    "giggolo":       {"saint":  12, "sinner": -21, "fumer":  -5, "tippler": -5},
    "vapeshape":     {"saint":  -5, "sinner":  14, "fumer":  21, "tippler": -10},
}

# 5) Decision rule using expected value (EV) computed from probabilities;
#    If probabilities are missing, fall back to deterministic persona pricing.
def ev_from_proba(p_s: float, p_d: float, table: dict[str, float]) -> float:
    # Joint persona probabilities under independence assumption:
    #   P(saint)  = (1 - p_s) * (1 - p_d)
    #   P(fumer)  =  p_s      * (1 - p_d)
    #   P(tippler)= (1 - p_s) *  p_d
    #   P(sinner) =  p_s      *  p_d
    p = {
        "saint":   (1 - p_s) * (1 - p_d),
        "fumer":    p_s      * (1 - p_d),
        "tippler": (1 - p_s) *  p_d,
        "sinner":   p_s      *  p_d,
    }
    # Expected value = sum over personas of P(persona) * price(persona)
    return sum(p[k] * table[k] for k in p)

# Vectorised advertiser decisions — EV > 0 → sell (1), otherwise exclude (0).
# have_proba is defined here so it is available for both the decision loop and
# the profit calculation below (single source of truth).
have_proba = df["proba_smokes"].notna() & df["proba_drinks"].notna()

for adv, table in PRICES.items():
    # EV for rows with full probability estimates; deterministic persona price as fallback.
    ev_vec = np.where(
        have_proba,
        [ev_from_proba(float(ps), float(pd_), table)
         for ps, pd_ in zip(df["proba_smokes"], df["proba_drinks"])],
        [table[p] for p in df["persona"]],
    )
    df[adv] = (ev_vec > 0).astype(int)

# 6) Expected Profits per advertiser (uses probabilities + decisions)
#    EV columns compute expected value per user; Profit_* multiplies EV by the sell flag.
print(f"[Q2] Row: {len(df)} | complete prob: {have_proba.sum()} ({have_proba.mean():.1%})")

profits = {}
for adv, table in PRICES.items():
    df[f"EV_{adv}"] = np.where(
        have_proba,
        [ev_from_proba(p_s, p_d, table) for p_s, p_d in zip(df["proba_smokes"], df["proba_drinks"])],
        0.0
    )
    # Profit counts only when the user was sold to that advertiser
    df[f"Profit_{adv}"] = df[adv].astype(int) * df[f"EV_{adv}"]
    profits[adv] = {
        "sold_count":           int(df[adv].sum()),
        "expected_profit_sum":  float(df[f"Profit_{adv}"].sum()),
        "expected_profit_mean": float(df[f"Profit_{adv}"].mean()),
    }

# Compact console report for quick inspection
print("\n=== Expected Profits ===")
for adv, vals in profits.items():
    print(
        f"{adv:15s} user sold count={vals['sold_count']:6d}  "
        f"total profits={vals['expected_profit_sum']:.2f}  "
        f"avg/user={vals['expected_profit_mean']:.4f}"
    )

# 7) Write back into the skeleton (in-place update) + optional final copies/exports
#    Keep the original skeleton shape/order
cols_order = ["device_id", "user_drinks", "user_smokes", "johnny_talker", "giggolo", "vapeshape", "persona"]
# Preserve any extra columns that might be in the skeleton already
if set(cols_order).issubset(df.columns):
    df_out = df[cols_order].copy()
else:
    df_out = df.copy()

updated_records = df_out.to_dict(orient="records")

# Overwrite original skeleton
with open(sk_path, "w", encoding="utf-8") as f:
    json.dump(updated_records, f, ensure_ascii=False, indent=2)
print(f"[Q2] Skeleton updated: {sk_path}")

# Optional: write a final copy into outputs_Q2 for submission/audit
final_copy = OUT / "classification_answers_with_personas.json"
with open(final_copy, "w", encoding="utf-8") as f:
    json.dump(updated_records, f, ensure_ascii=False, indent=2)
print(f"[Q2] Final copy: {final_copy}")

# Optional: detailed CSV export for audit/reporting (includes probabilities, EVs, profits)
df.to_csv(OUT / "profits_detailed.csv", index=False)


[Q2] Row: 10000 | complete prob: 10000 (100.0%)

=== Expected Profits ===
johnny_talker   user sold count=  4419  total profits=47912.83  avg/user=4.7913
giggolo         user sold count=  5797  total profits=43532.55  avg/user=4.3533
vapeshape       user sold count=  3477  total profits=31308.52  avg/user=3.1309
[Q2] Skeleton updated: outputs_setup/classification_skeleton.json
[Q2] Final copy: outputs_Q2/classification_answers_with_personas.json
